# 📥 BlueStock Mutual Fund Analytics — 01. Data Ingestion & Audit
**Capstone Deliverable D1/D3 Prerequisite**  
**Author:** BlueStock Quantitative Analytics Team  
**Scope:** 10 Raw Datasets | AMFI Scheme Master | Historical NAVs | Investor Transactions | Holdings | Benchmarks

---

### Analytical Objectives
1. **Automated Discovery:** Scan and catalog all raw CSV files in `data/raw/`.
2. **Schema & Dimensionality Audit:** Inspect row counts, column types, and structural properties.
3. **Missing Value Profiling:** Calculate null counts and missingness percentages per column.
4. **Duplicate Record Detection:** Identify duplicate rows and key-level collisions.
5. **Data Health Matrix:** Generate an executive data health scorecard and export to `reports/day1_data_quality_summary.csv`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Set project paths using pathlib (avoiding hardcoded paths)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
REPORTS_DIR = BASE_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Raw Data Directory: {RAW_DIR}")


Base Directory: C:\Users\sriha\OneDrive\Desktop\Bluestock-MF-Analytics
Raw Data Directory: C:\Users\sriha\OneDrive\Desktop\Bluestock-MF-Analytics\data\raw


### 1. Catalog Raw Datasets

In [2]:
raw_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(raw_files)} CSV files in data/raw/:")
for f in raw_files:
    size_kb = f.stat().st_size / 1024
    print(f"  • {f.name:<35} ({size_kb:>8.2f} KB)")


Found 18 CSV files in data/raw/:
  • 01_fund_master.csv                  (    6.66 KB)
  • 02_nav_history.csv                  ( 1190.99 KB)
  • 03_aum_by_fund_house.csv            (    3.92 KB)
  • 04_monthly_sip_inflows.csv          (    1.63 KB)
  • 05_category_inflows.csv             (    3.58 KB)
  • 06_industry_folio_count.csv         (    0.80 KB)
  • 07_scheme_performance.csv           (    6.44 KB)
  • 08_investor_transactions.csv        ( 3054.61 KB)
  • 09_portfolio_holdings.csv           (   23.35 KB)
  • 10_benchmark_indices.csv            (  245.09 KB)
  • live_nav_118632.csv                 (  423.93 KB)
  • live_nav_119092.csv                 (  405.39 KB)
  • live_nav_119551.csv                 (  510.00 KB)
  • live_nav_120503.csv                 (  385.51 KB)
  • live_nav_120841.csv                 (  365.74 KB)
  • live_nav_125497.csv                 (  330.73 KB)
  • live_nav_fetch_summary.csv          (    0.75 KB)
  • scheme_code_search_results.csv      (    2.18

### 2. Comprehensive Schema, Nulls & Duplicates Audit

In [3]:
inspection_records = []

for file_path in raw_files:
    try:
        df = pd.read_csv(file_path)
        missing_count = int(df.isna().sum().sum())
        total_cells = df.shape[0] * df.shape[1]
        missing_pct = round((missing_count / total_cells * 100), 2) if total_cells > 0 else 0.0
        dup_count = int(df.duplicated().sum())
        
        inspection_records.append({
            "File Name": file_path.name,
            "Rows": df.shape[0],
            "Columns": df.shape[1],
            "Missing Cells": missing_count,
            "Missing (%)": missing_pct,
            "Duplicate Rows": dup_count,
            "Status": "Passed"
        })
    except Exception as e:
        inspection_records.append({
            "File Name": file_path.name,
            "Rows": 0,
            "Columns": 0,
            "Missing Cells": 0,
            "Missing (%)": 0.0,
            "Duplicate Rows": 0,
            "Status": f"Failed: {e}"
        })

inspection_df = pd.DataFrame(inspection_records)
inspection_df.to_csv(REPORTS_DIR / "day1_dataset_inspection.csv", index=False)
display(inspection_df)


,File Name,Rows,Columns,Missing Cells,Missing (%),Duplicate Rows,Status
0,01_fund_master.csv,40,15,0,0.00,0,Passed
1,02_nav_history.csv,46000,3,0,0.00,0,Passed
2,03_aum_by_fund_house.csv,90,5,0,0.00,0,Passed
3,04_monthly_sip_inflows.csv,48,6,12,4.17,0,Passed
4,05_category_inflows.csv,144,3,0,0.00,0,Passed
5,06_industry_folio_count.csv,21,6,0,0.00,0,Passed
6,07_scheme_performance.csv,40,19,0,0.00,0,Passed
7,08_investor_transactions.csv,32778,13,0,0.00,0,Passed
8,09_portfolio_holdings.csv,322,8,0,0.00,0,Passed
9,10_benchmark_indices.csv,8050,3,0,0.00,0,Passed


### 3. Deep Dive: Key Primary Datasets Preview

In [4]:
# 1. Fund Master Sample
df_fund_master = pd.read_csv(RAW_DIR / "01_fund_master.csv")
print("=== Fund Master (01_fund_master.csv) ===")
display(df_fund_master.head(5))

# 2. Historical NAV Sample
df_nav = pd.read_csv(RAW_DIR / "02_nav_history.csv")
print("\n=== Historical NAV (02_nav_history.csv) ===")
print(f"Total NAV observations: {len(df_nav):,}")
print(f"Unique AMFI codes: {df_nav['amfi_code'].nunique()}")
display(df_nav.head(5))

# 3. Investor Transactions Sample
df_tx = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
print("\n=== Investor Transactions (08_investor_transactions.csv) ===")
print(f"Total transactions: {len(df_tx):,}")
display(df_tx.head(5))


=== Fund Master (01_fund_master.csv) ===


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02



=== Historical NAV (02_nav_history.csv) ===
Total NAV observations: 46,000
Unique AMFI codes: 40


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692



=== Investor Transactions (08_investor_transactions.csv) ===
Total transactions: 32,778


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


### 4. Summary & Data Quality Takeaways
- **10 Primary Datasets Ingested:** Encompassing scheme attributes, daily historical NAVs, retail/HNI transactions, category flows, and Nifty benchmarks.
- **Key Identifiers:** `amfi_code` serves as the primary relational key linking fund master, historical NAVs, performance metrics, and investor transactions.
- **Next Steps:** Proceed to `02_data_cleaning.ipynb` for date standardization, forward-filling NAV calendar series across holidays/weekends, and schema normalization.
